# Fase 4 — Clasificación neuronal con PySpark MLlib
## MLP (Multilayer Perceptron) sobre vectores TF-IDF

**Entrada:** vectores TF-IDF de MIND Large (Fase 2) — 18 categorías etiquetadas  
**Salida:** modelo MLP entrenado + predicciones

**Arquitectura:**
```
Entrada (65536) → Oculta 1 (256, ReLU) → Oculta 2 (128, ReLU) → Salida (N_clases, Softmax)
```

## 0. Rutas

In [ ]:
import os

PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATOS_PATH   = os.path.join(PROJECT_PATH, "datos")
MODELS_PATH  = os.path.join(PROJECT_PATH, "models")

TFIDF_MIND   = os.path.join(DATOS_PATH,  "processed", "tfidf_mind")
OUT_MODEL    = os.path.join(MODELS_PATH, "mlp_clasificador")
OUT_PREDS    = os.path.join(DATOS_PATH,  "processed", "predicciones")

os.makedirs(MODELS_PATH, exist_ok=True)
os.makedirs(OUT_PREDS,   exist_ok=True)

print(f"PROJECT_PATH → {PROJECT_PATH}")
print(f"TFIDF_MIND   → {TFIDF_MIND}  ({'OK' if os.path.exists(TFIDF_MIND) else 'FALTA'})")

## 1. Dependencias

In [ ]:
# Ejecuta solo la primera vez
# !pip install pyspark

## 2. Inicialización de PySpark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Clasificador-MLP")
    .master("local[2]")
    .config("spark.driver.memory", "6g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"PySpark {spark.version} listo.")

## 3. Carga de datos (salida de Fase 2)

In [ ]:
df = spark.read.parquet(TFIDF_MIND)

print(f"Total documentos: {df.count():,}")
print(f"Columnas: {df.columns}")

print("\nDistribución de categorías:")
df.groupBy("label").count().orderBy(F.desc("count")).show(20, truncate=False)

## 4. Codificación de etiquetas

El MLP de MLlib requiere etiquetas numéricas. `StringIndexer` convierte las categorías
a índices enteros de forma determinista (ordena por frecuencia descendente).

In [ ]:
from pyspark.ml.feature import StringIndexer

indexer       = StringIndexer(inputCol="label", outputCol="label_idx", handleInvalid="skip")
indexer_model = indexer.fit(df)
df_indexed    = indexer_model.transform(df)

labels        = indexer_model.labels
N_CLASES      = len(labels)

print(f"Clases ({N_CLASES}):")
for i, l in enumerate(labels):
    print(f"  {i:2d} → {l}")

## 5. Split train / test

Usamos el campo `split` del Parquet (train vs dev) para respetar la partición original de MIND.  
Si no existe ese campo, hacemos split aleatorio 80/20.

In [ ]:
if "split" in df_indexed.columns:
    train_df = df_indexed.filter(F.col("split") == "train")
    test_df  = df_indexed.filter(F.col("split") == "dev")
    print("Split por campo 'split' original de MIND:")
else:
    train_df, test_df = df_indexed.randomSplit([0.8, 0.2], seed=42)
    print("Split aleatorio 80/20:")

print(f"  Train: {train_df.count():,} docs")
print(f"  Test:  {test_df.count():,} docs")

## 6. Entrenamiento del MLP

**Arquitectura:** `[65536, 256, 128, N_clases]`
- Capas ocultas con activación **ReLU**
- Capa de salida con activación **Softmax**
- Optimizador: L-BFGS (default en MLlib) — converge más rápido que SGD en local

PySpark MLlib distribuye el entrenamiento particionando los datos entre workers.

In [ ]:
import time
from pyspark.ml.classification import MultilayerPerceptronClassifier

NUM_FEATURES = 65536  # debe coincidir con el valor usado en Fase 2

capas = [NUM_FEATURES, 256, 128, N_CLASES]

mlp = MultilayerPerceptronClassifier(
    featuresCol="tfidf_vector",
    labelCol="label_idx",
    predictionCol="prediccion_idx",
    layers=capas,
    blockSize=128,
    maxIter=50,
    seed=42,
)

print(f"Arquitectura MLP: {capas}")
print("Entrenando... (puede tardar 10–30 min en local)")
t0 = time.time()

mlp_model = mlp.fit(train_df)

print(f"Entrenamiento completado en {(time.time()-t0)/60:.1f} min")

## 7. Predicciones y decodificación de etiquetas

In [ ]:
from pyspark.ml.feature import IndexToString

converter = IndexToString(
    inputCol="prediccion_idx",
    outputCol="prediccion",
    labels=labels
)

preds_train = converter.transform(mlp_model.transform(train_df))
preds_test  = converter.transform(mlp_model.transform(test_df))

print("Ejemplos de predicción (test):")
preds_test.select("doc_id", "label", "prediccion").show(10, truncate=False)

## 8. Evaluación — Accuracy y F1

Reportamos **accuracy global** y **F1 ponderado** sobre el conjunto de test,
tal como requiere la Fase 5 del pipeline.

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label_idx", predictionCol="prediccion_idx", metricName="accuracy"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label_idx", predictionCol="prediccion_idx", metricName="weightedFMeasure"
)

acc_train = evaluator_acc.evaluate(mlp_model.transform(train_df))
acc_test  = evaluator_acc.evaluate(mlp_model.transform(test_df))
f1_test   = evaluator_f1.evaluate(mlp_model.transform(test_df))

print("=" * 40)
print("RESULTADOS DEL CLASIFICADOR MLP")
print("=" * 40)
print(f"  Accuracy train : {acc_train:.4f}")
print(f"  Accuracy test  : {acc_test:.4f}")
print(f"  F1 ponderado   : {f1_test:.4f}")
print("=" * 40)

## 9. F1 por clase

Calculamos F1 por cada categoría para identificar clases difíciles — requerido en Fase 5.

In [ ]:
import pandas as pd

preds_pd = (
    mlp_model.transform(test_df)
    .select("label_idx", "prediccion_idx")
    .toPandas()
)

from sklearn.metrics import classification_report

print(classification_report(
    preds_pd["label_idx"].astype(int),
    preds_pd["prediccion_idx"].astype(int),
    target_names=labels
))

## 10. Guardado del modelo y predicciones

In [ ]:
# Modelo MLP
mlp_model.save(OUT_MODEL)
print(f"Modelo guardado: {OUT_MODEL}")

# Predicciones del test para Fase 5
(
    preds_test
    .select("doc_id", "label", "prediccion", "label_idx", "prediccion_idx")
    .write.mode("overwrite")
    .parquet(OUT_PREDS)
)
print(f"Predicciones guardadas: {OUT_PREDS}")

## 11. Análisis costo-comunicación — Fase 4

Conexión con el modelo teórico del curso.

In [ ]:
n_train = train_df.count()
n_test  = test_df.count()

print("MODELO COSTO-COMUNICACIÓN — CLASIFICADOR MLP (Fase 4)")
print("-" * 55)
print(f"Documentos train : {n_train:,}")
print(f"Documentos test  : {n_test:,}")
print(f"Arquitectura     : {capas}")
print(f"Parámetros totales: {sum(capas[i]*capas[i+1] + capas[i+1] for i in range(len(capas)-1)):,}")
print()
print("Costo por etapa:")
print(f"  Forward pass (1 doc): O(Σ w_i · h_i) donde w_i=neuronas capa i")
print(f"  Forward pass (N docs): O(N · Σ w_i · h_i) — paralelo en Spark")
print(f"  Backward pass (L-BFGS): O(iteraciones · N · parámetros)")
print(f"  Comunicación: gradientes se agregan entre particiones — O(parámetros)")
print()
print("Ventaja de distribución en Spark MLlib:")
print("  - Datos particionados entre workers → forward/backward en paralelo")
print("  - Solo los gradientes viajan por la red, no los datos")
print(f"  - Reducción de comunicación: O(parámetros) << O(N · features)")

In [ ]:
spark.stop()
print("SparkSession cerrada. Fase 4 completada.")
print("Siguiente paso: 05_evaluacion.ipynb")